In [59]:
%pip install -q sentence-transformers transformers accelerate Pillow


Note: you may need to restart the kernel to use updated packages.


In [60]:
import os
import json
from sentence_transformers import SentenceTransformer, util
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
import torch
import requests
import base64
# Load your template metadata
with open("templates.json", "r") as f:
    templates = json.load(f)


In [61]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")
descs = [t["description"] + " " + " ".join(t["tags"]) for t in templates]
desc_embeddings = st_model.encode(descs, convert_to_tensor=True)


In [62]:
def select_template(prompt):
    pe = st_model.encode(prompt, convert_to_tensor=True)
    sims = util.cos_sim(pe, desc_embeddings)[0]
    idx = sims.argmax().item()
    tmpl = templates[idx]
    return os.path.join("templates", tmpl["id"]), tmpl


In [63]:
def generate_caption(image_path, prompt, template):
    # Read image as base64
    with open(image_path, "rb") as img_file:
        image_data = base64.b64encode(img_file.read()).decode("utf-8")

    # Construct contextual prompt
    context_prompt = f"""
Meme idea: {prompt}
Template description: {template['description']}
Tags: {', '.join(template['tags'])}

Generate a short, funny, and sarcastic meme caption for this template. Avoid literal explanations.
"""

    # POST to LLaVA-
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llava:7b",
            "messages": [
                {"role": "system", "content": "You're a meme expert. Generate a short, funny, and highly shareable caption that fits the meme template based on the given context. Avoid being literal — go for irony, exaggeration, or punchline format most importantly be funny."},

                
                {"role": "user", "content": context_prompt}
            ],
            "images": [image_data],
            "stream": False
        }
    )

    response.raise_for_status()
    caption = response.json()["message"]["content"]
    return caption.strip()


In [64]:
from PIL import Image, ImageDraw, ImageFont
import textwrap
import os

def render_caption_on_template(image_path, caption_text, output_path="output_meme.jpg"):
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    try:
        font_path = "arialbd.ttf"  # Windows bold font
        font = ImageFont.truetype(font_path, size=30)
    except:
        font = ImageFont.load_default()

    img_width, img_height = image.size

    # Wrap text
    margin = 30
    max_text_width = img_width - 2 * margin
    wrapper = textwrap.TextWrapper(width=30)
    wrapped_text = wrapper.fill(text=caption_text)

    # Get text size using textbbox
    bbox = draw.textbbox((0, 0), wrapped_text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x = (img_width - text_width) / 2
    y = img_height - text_height - 30

    # Draw black shadow
    shadow_offset = 2
    for dx in [-shadow_offset, shadow_offset]:
        for dy in [-shadow_offset, shadow_offset]:
            draw.text((x + dx, y + dy), wrapped_text, font=font, fill="black")

    # Draw white text
    draw.text((x, y), wrapped_text, font=font, fill="white")

    image.save(output_path)
    print(f"Meme saved to: {output_path}")
    return output_path



In [69]:
prompt = input("Enter your meme prompt: ")
img_path, tmpl = select_template(prompt)
print(img_path)
print(tmpl)
print("\n Using template:", tmpl["id"])
print(" Tags:", ", ".join(tmpl["tags"]), "\n")

caption = generate_caption(img_path, prompt, tmpl)

print(" Generated Caption:", caption)
render_caption_on_template(img_path, caption)

templates\roll_safe.jpg
{'id': 'roll_safe.jpg', 'tags': ['sarcasm', 'logic', 'clever', 'bad decisions', 'irony'], 'caption_style': 'top-bottom', 'description': 'Man pointing to head as if being clever. Used to show faulty logic or smart-sounding nonsense.', 'text': 'inside'}

 Using template: roll_safe.jpg
 Tags: sarcasm, logic, clever, bad decisions, irony 

 Generated Caption: "When someone asks if politicians are corrupt, just point to your head. It's where all the smart, unbiased decision-making happens."
Meme saved to: output_meme.jpg


'output_meme.jpg'

In [68]:

image = Image.open("C:\\Users\\ratan\\OneDrive\\Desktop\\meme-bot\\output_meme.jpg")
image.show()